# SST DRVI factors — pseudo-progression

Do the factors track Alzheimer's severity, and if so, *how*? `01_qc.ipynb` settles whether
each fit is trustworthy and `02_explore.ipynb` what each factor means; this notebook takes
the interpreted factors and asks one question of them — does a factor direction move with
SEA-AD's continuous pseudo-progression score (CPS), and is that movement a change in what
the surviving SST neurons are doing or just the disappearance of the vulnerable ones?

Three strands meet here:

1. **Split factor directions.** `DR n+` and `DR n-` are separate programs, so they are
   tested separately (`02_explore.ipynb`, section 4).
2. **CPS.** `CPS_Global` is one value per donor and `CPS_Local` one per donor × region —
   *not* per cell. Section 1 establishes that before anything is correlated.
3. **The existing scCODA results** under `/data/multiregion/pertpy/CPS_Local/`, a finished
   compositional analysis of all 174 supertypes against `CPS_Local`. Nothing here refits
   it; `/data` is read-only and this notebook only reads what is already there.

CPU-only. Kernel: `sst-drvi`.

In [ ]:
import json
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy import stats

import sst_drvi as S

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(4, 4))
pd.set_option("display.max_rows", 140)

## Helpers used more than once

Everything else in this notebook is written out where it is used. These four have several
callers each — `cps_trend` alone is called seven times — so they are named here instead of
repeated, along with the three small numeric pieces they share.

In [ ]:
# These are the only helpers with more than one caller below; everything else in this
# notebook is written out where it is used. `logger` is wired up because the functions
# report what they drop -- units under the cell floor, directions that are constant -- and
# those lines are worth seeing rather than swallowing.
import logging
from collections.abc import Sequence

logger = logging.getLogger("progression")
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())
    logger.setLevel(logging.INFO)


def split_by_sign(embed, hide_vanished: bool = True) -> pd.DataFrame:
    """Per-cell activation magnitude of each factor *direction*.

    DRVI factors are directional: ``DR n+`` and ``DR n-`` can encode unrelated programs,
    which is why the interpretability scores are computed per direction. A signed factor
    value conflates the two, so an association carried by only one direction is diluted
    by the cells sitting on the other side of zero. This returns ``relu(x)`` and
    ``relu(-x)`` as separate non-negative columns.

    Columns are labelled and ordered exactly like `interpretability_scores`
    (``DR 1+``, ``DR 1-``, ``DR 2+``, ...), so the two tables join on them. Directions
    flagged vanished by ``set_latent_dimension_stats`` are dropped by default.
    """
    x = np.asarray(embed.X, dtype=np.float32)
    info = embed.var
    columns = {}
    for pos, (_, row) in enumerate(info.iterrows()):
        values = x[:, pos]
        for direction, magnitude, dead in (
            ("+", np.maximum(values, 0.0), row["vanished_positive_direction"]),
            ("-", np.maximum(-values, 0.0), row["vanished_negative_direction"]),
        ):
            if hide_vanished and dead:
                continue
            columns[f"{row['title']}{direction}"] = magnitude

    out = pd.DataFrame(columns, index=embed.obs_names)
    order = (
        pd.concat([info.assign(direction="+"), info.assign(direction="-")])
        .assign(title=lambda df: df["title"] + df["direction"])
        .sort_values(["order", "direction"])["title"]
    )
    return out[[c for c in order if c in out.columns]]


def pseudobulk_directions(
    embed,
    keys: Sequence[str] = ("Donor ID", "Brain Region"),
    hide_vanished: bool = True,
    min_cells: int = 20,
    carry: Sequence[str] = (),
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Mean activity of each factor *direction* per unit of analysis.

    Returns ``(activity, meta)`` sharing one index: ``activity`` is units x directions
    (mean of the non-negative ``relu`` magnitudes from `split_by_sign`), ``meta``
    carries ``n_cells`` plus the first value of every column in ``carry``.

    The mean is over cells, so a unit built from 40 nuclei is noisier than one built from
    4,000 and ``min_cells`` drops the worst of them; ``n_cells`` is returned so the rest
    can be weighted or inspected rather than silently trusted.
    """
    split = split_by_sign(embed, hide_vanished=hide_vanished)
    obs = embed.obs
    keys = list(keys)
    missing = [k for k in [*keys, *carry] if k not in obs]
    if missing:
        raise KeyError(f"obs is missing {missing}")

    grouper = [obs[k] for k in keys]
    activity = split.groupby(grouper, observed=True).mean()
    meta = pd.DataFrame({"n_cells": obs.groupby(grouper, observed=True).size()})
    if carry:
        meta = meta.join(obs.groupby(grouper, observed=True)[list(carry)].first())

    keep = meta["n_cells"] >= min_cells
    if not keep.all():
        logger.info(
            "dropping %d of %d %s units with fewer than %d cells",
            (~keep).sum(),
            len(keep),
            " x ".join(keys),
            min_cells,
        )
    activity, meta = activity[keep.to_numpy()], meta[keep]

    # A score that varies inside a unit was aggregated at the wrong level; say so rather
    # than averaging it away, since the whole point of the unit is that it is constant.
    for col in carry:
        if col in S.CPS_COLUMNS:
            spread = (
                pd.to_numeric(obs[col], errors="coerce")
                .groupby(grouper, observed=True)
                .nunique()
                .max()
            )
            if spread > 1:
                logger.warning(
                    "%s takes up to %d values within a single %s unit; "
                    "meta holds only the first",
                    col,
                    spread,
                    " x ".join(keys),
                )
    return activity, meta


def bh_fdr(p: np.ndarray) -> np.ndarray:
    """Benjamini-Hochberg adjusted p-values, NaN-safe."""
    p = np.asarray(p, dtype=float)
    out = np.full(p.shape, np.nan)
    finite = np.isfinite(p)
    values = p[finite]
    if values.size == 0:
        return out
    order = np.argsort(values)
    ranked = values[order] * values.size / np.arange(1, values.size + 1)
    adjusted = np.minimum.accumulate(ranked[::-1])[::-1].clip(max=1.0)
    result = np.empty_like(values)
    result[order] = adjusted
    out[finite] = result
    return out


def spearman_columns(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Spearman rho of every column of ``x`` against ``y``, vectorized.

    Pearson on average ranks, which is what Spearman is. Written out rather than looped
    through ``scipy.stats.spearmanr`` because the cluster bootstrap in
    `cps_trend_clustered` calls it thousands of times.
    """
    from scipy.stats import rankdata

    xr = rankdata(x, axis=0).astype(np.float64)
    yr = rankdata(y).astype(np.float64)
    xr -= xr.mean(axis=0)
    yr -= yr.mean()
    denominator = np.sqrt((xr**2).sum(axis=0) * (yr**2).sum())
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(denominator > 0, (xr * yr[:, None]).sum(axis=0) / denominator, np.nan)


def cps_trend(
    activity: pd.DataFrame,
    meta: pd.DataFrame,
    cps: str = "CPS_Local",
    by: str | Sequence[str] | None = None,
    min_units: int = 10,
) -> pd.DataFrame:
    """Signed Spearman of every factor direction against a CPS score, across units.

    One row per direction (per level of ``by``, when given) with ``rho``, ``p``, the
    Benjamini-Hochberg ``q`` computed within that level, and the unit count behind it.

    Unlike `factor_association`, which reports ``|rho|`` so a heatmap can rank
    covariates, this keeps the **sign**: the question here is whether a direction rises
    or falls with pseudo-progression, and that is the entire finding.

    ``by="Brain Region"`` is the form to trust for ``CPS_Local``. Pooling regions mixes a
    within-donor contrast with a between-donor one -- a region where SST nuclei are
    scarce and CPS is high would produce a trend out of the regional composition alone --
    whereas within one region every donor contributes exactly one unit. Passing several
    columns (``["Brain Region", "Supertype"]``, over units built with ``Supertype`` in
    ``keys``) strips composition out entirely: cell type is then fixed within a stratum,
    so a surviving trend is a change of state rather than a change of abundance.
    """
    from scipy import stats

    if cps not in meta:
        raise KeyError(f"{cps!r} is not in meta; pass it through pseudobulk_directions(carry=...)")

    by_cols = [by] if isinstance(by, str) else list(by) if by is not None else []
    groups = (
        [((), meta.index)]
        if not by_cols
        else [
            (level if isinstance(level, tuple) else (level,), idx)
            for level, idx in meta.groupby(by_cols, observed=True).groups.items()
        ]
    )

    rows = []
    for level, index in groups:
        values = pd.to_numeric(meta.loc[index, cps], errors="coerce")
        keep = index[values.notna().to_numpy()]
        if len(keep) < min_units:
            logger.debug("skipping %s: %d units, below min_units=%d", level, len(keep), min_units)
            continue
        x = activity.loc[keep]
        # Constant columns make Spearman undefined; a direction silent in this region is
        # a real answer ("no cells drive it here"), not an error.
        live = x.nunique() > 1
        n_live = int(live.sum())
        if n_live == 0:
            logger.debug("skipping %s: every direction is constant across its units", level)
            continue
        result = stats.spearmanr(x.loc[:, live].to_numpy(), values[keep].to_numpy())
        # Two columns in, and scipy returns scalars rather than a correlation matrix.
        if n_live == 1:
            rho, p = np.array([result.statistic]), np.array([result.pvalue])
        else:
            rho = np.atleast_2d(result.statistic)[:-1, -1]
            p = np.atleast_2d(result.pvalue)[:-1, -1]
        frame = pd.DataFrame(
            {"direction": x.columns[live], "n_units": len(keep), "rho": rho, "p": p}
        )
        for position, column in enumerate(by_cols):
            frame.insert(position, column, level[position])
        frame["q"] = bh_fdr(frame["p"].to_numpy())
        rows.append(frame)

    if not rows:
        raise ValueError(
            f"nothing testable: no group had {min_units}+ units with a finite {cps} "
            "and at least one non-constant direction"
        )
    out = pd.concat(rows, ignore_index=True)
    return out.sort_values([*by_cols, "q"]).reset_index(drop=True)


def cps_trend_consensus(stratified: pd.DataFrame, q_threshold: float = 0.1) -> pd.DataFrame:
    """Collapse a stratified `cps_trend` into one row per direction.

    Strata share donors, so their tests are not independent and the p-values must not be
    combined into one. What *is* readable is reproducibility: a direction whose sign
    agrees across ten regions is telling a different story from one carried by a single
    region, however small that region's q.
    """
    grouped = stratified.groupby("direction", observed=True)
    out = pd.DataFrame(
        {
            "n_strata": grouped.size(),
            "median_rho": grouped["rho"].median(),
            "min_q": grouped["q"].min(),
            "n_negative": grouped["rho"].apply(lambda r: int((r < 0).sum())),
            "n_positive": grouped["rho"].apply(lambda r: int((r > 0).sum())),
            "n_significant": grouped["q"].apply(lambda q: int((q < q_threshold).sum())),
        }
    )
    out["sign_agreement"] = out[["n_negative", "n_positive"]].max(axis=1) / out["n_strata"]
    return out.sort_values(["n_significant", "median_rho"], key=abs, ascending=False)


def cps_trend_clustered(
    activity: pd.DataFrame,
    meta: pd.DataFrame,
    cps: str = "CPS_Local",
    cluster: str = "Donor ID",
    n_boot: int = 1000,
    alpha: float = 0.05,
    random_state: int = 0,
) -> pd.DataFrame:
    """Pooled Spearman over every unit, with a donor-cluster bootstrap interval.

    This is the analysis that treats each donor x region as its own observation: no
    stratification, no collapsing, all 516 rows at once. It is the view with the most
    power and the one most often reported, and it has two distinct weaknesses that should
    not be conflated.

    **Pseudoreplication** is repaired here. The rows come from far fewer donors than there
    are rows, so the textbook p-value is anticonservative -- it counts 516 independent
    pieces of evidence where there are 84. Resampling *donors* with replacement and
    recomputing the correlation on whatever units those donors bring propagates the real
    unit of replication into the interval. ``ci_low``/``ci_high`` are the percentile
    interval over those resamples and ``robust`` is whether it clears zero; ``p`` and
    ``q`` are kept beside them, labelled naive, so the gap between the two is visible.

    **The region confound is not repaired**, by this or anything else at this unit:
    pooling still mixes a between-region contrast with a between-donor one, which is why
    `cps_trend` with ``by="Brain Region"`` exists. Read ``rho`` against that
    function's per-region consensus rather than instead of it.
    """
    if cps not in meta:
        raise KeyError(f"{cps!r} is not in meta; pass it through pseudobulk_directions(carry=...)")

    values = pd.to_numeric(meta[cps], errors="coerce")
    keep = values.notna().to_numpy()
    x = activity.loc[keep].to_numpy(dtype=np.float64)
    y = values[keep].to_numpy(dtype=np.float64)
    clusters = pd.Series(meta.index.get_level_values(cluster), index=meta.index)[keep]

    live = x.std(axis=0) > 0
    if not live.any():
        raise ValueError("every direction is constant across the units")

    rho = np.full(x.shape[1], np.nan)
    rho[live] = spearman_columns(x[:, live], y)

    # scipy's asymptotic t approximation, the p-value the pooled test would report if the
    # rows really were independent -- kept to be compared against, not to be used alone.
    from scipy import stats

    n = len(y)
    with np.errstate(invalid="ignore", divide="ignore"):
        t = rho * np.sqrt((n - 2) / np.clip(1 - rho**2, 1e-12, None))
    p = 2 * stats.t.sf(np.abs(t), df=n - 2)

    rows = [np.flatnonzero(clusters.to_numpy() == name) for name in pd.unique(clusters)]
    rng = np.random.default_rng(random_state)
    replicates = np.empty((n_boot, int(live.sum())))
    for draw in range(n_boot):
        picked = rng.integers(0, len(rows), len(rows))
        index = np.concatenate([rows[i] for i in picked])
        replicates[draw] = spearman_columns(x[np.ix_(index, np.flatnonzero(live))], y[index])

    bounds = np.full((2, x.shape[1]), np.nan)
    bounds[:, live] = np.nanpercentile(
        replicates, [100 * alpha / 2, 100 * (1 - alpha / 2)], axis=0
    )

    out = pd.DataFrame(
        {
            "direction": activity.columns,
            "n_units": n,
            "n_donors": len(rows),
            "rho": rho,
            "p_naive": p,
            "ci_low": bounds[0],
            "ci_high": bounds[1],
        }
    ).dropna(subset=["rho"])
    out["q_naive"] = bh_fdr(out["p_naive"].to_numpy())
    out["robust"] = (out["ci_low"] > 0) | (out["ci_high"] < 0)
    logger.info(
        "%s: %d units from %d donors; %d of %d directions at naive q < 0.1, "
        "%d with a %g%% donor-bootstrap interval clear of zero",
        cps,
        n,
        len(rows),
        int((out["q_naive"] < 0.1).sum()),
        len(out),
        int(out["robust"].sum()),
        100 * (1 - alpha),
    )
    return out.sort_values("q_naive").reset_index(drop=True)

## The fit being read

Same preference order as `02_explore.ipynb`: the converged covariate refit first, then the
200-epoch covariate fit, then the baseline. Set `RUN` directly to override.

In [ ]:
CANDIDATES = ["sst_k64_covar_long", "sst_k64_covar", "sst_k64"]
MODEL = "drvi"

RUN = None
for candidate in CANDIDATES:
    embed_path = S.EMBED_DIR / f"{candidate}_{MODEL}_embed.h5ad"
    if embed_path.exists():
        RUN = candidate
        break
    print(f"{candidate} not on disk")
if RUN is None:
    raise FileNotFoundError(f"no DRVI embedding found among {CANDIDATES}")

embed = ad.read_h5ad(embed_path)
SUPERTYPES = list(embed.obs["Supertype"].cat.categories)

print(f"run: {RUN}\n{embed.n_obs:,} nuclei x {embed.n_vars} factors, "
      f"{len(SUPERTYPES)} supertypes, {embed.obs['Donor ID'].nunique()} donors")
print("run:    ", dict(embed.uns["drvi_run"]))

## 1. What CPS measures, and at what *n*

The CPS columns sit in `obs`, one value per nucleus, which makes it easy to treat them as
per-cell measurements. They are not. Each is a donor-level (`CPS_Global`) or donor ×
region-level (`CPS_Local`) quantity copied down onto every nucleus of that donor, so a
correlation over 231,107 rows is testing 84 or ~530 independent units with an *n* two
orders of magnitude too large.

`cps_resolution` checks this rather than assuming it — the column that matters is the
number of distinct values a score takes *within* a donor, and within a donor × region.

In [ ]:
# The column that matters is how many distinct values a score takes *within* one donor,
# and within one donor x region. A 1 means the score is a measurement of that unit, so its
# effective sample size is the unit count -- not the 231,107 nuclei it was copied onto.
KEYS = ["Donor ID", "Brain Region"]

resolution = {}
for column in S.CPS_COLUMNS:
    values = pd.to_numeric(embed.obs[column], errors="coerce")
    row = {"n_cells": int(values.notna().sum()), "n_distinct": int(values.nunique())}
    for depth in range(1, len(KEYS) + 1):
        nested = KEYS[:depth]
        row[f"max_per_{' x '.join(nested)}"] = int(
            values.groupby([embed.obs[key] for key in nested], observed=True).nunique().max()
        )
    resolution[column] = row

pd.DataFrame(resolution).T

`CPS_Global` is constant within a donor (84 distinct values, 84 donors); `CPS_Local` varies
by region within a donor but is constant within a donor × region (up to 9 values per donor
— one per region sampled). So the units of analysis are **84 donors** and **530 donor ×
regions**, and everything below aggregates to one of those before testing.

### What the per-cell version would have claimed

Both columns rank the factor directions, but only one of them has an honest *n*. The
per-cell association is the one `02_explore.ipynb` reports (as `|rho|`, for its heatmap);
the per-unit one is the same statistic computed on donor × region means.

In [ ]:
activity, meta = pseudobulk_directions(
    embed, keys=("Donor ID", "Brain Region"),
    carry=S.CPS_COLUMNS + ("Cognitive Status", "Braak", "Sex", "Age at Death"),
)
print(f"{activity.shape[0]} donor x region units x {activity.shape[1]} live directions")
print(f"median nuclei per unit: {meta['n_cells'].median():.0f} "
      f"(min {meta['n_cells'].min()}, max {meta['n_cells'].max()})")

# The per-cell comparison, written out: |Spearman| of each direction's activation
# against the CPS value copied onto every nucleus. One call, so it lives here rather than
# behind a name -- and it exists only to be compared against the per-unit column.
per_cell_split = split_by_sign(embed)
per_cell_cps = pd.to_numeric(embed.obs["CPS_Local"], errors="coerce").to_numpy(float)
finite = np.isfinite(per_cell_cps)
per_cell = pd.Series(
    np.abs(np.atleast_2d(stats.spearmanr(per_cell_split.to_numpy()[finite],
                                         per_cell_cps[finite]).statistic)[:-1, -1]),
    index=per_cell_split.columns,
)
pooled = cps_trend(activity, meta, cps="CPS_Local").set_index("direction")

inflation = pd.DataFrame({
    "per_cell_abs_rho": per_cell,
    "per_unit_abs_rho": pooled["rho"].abs(),
    "per_unit_q": pooled["q"],
}).dropna().sort_values("per_cell_abs_rho", ascending=False)
inflation["rank_per_cell"] = inflation["per_cell_abs_rho"].rank(ascending=False).astype(int)
inflation["rank_per_unit"] = inflation["per_unit_abs_rho"].rank(ascending=False).astype(int)
print(f"\nn behind each column: {embed.n_obs:,} cells vs {len(activity)} units")
inflation.head(12).round(3)

The effect sizes are not the problem — averaging over cells mostly *raises* `|rho|`, because
the per-cell version is diluted by within-unit noise. The problem is the *p*-value attached
to them, and the reordering: the direction the per-cell test ranks first is rarely the one
the per-unit test ranks first. Only the per-unit column is used from here on.

## 2. Factor directions against `CPS_Local`, region by region

Pooling all 516 units mixes two different contrasts: donors differ in CPS, and so do
regions within a donor. A region where SST nuclei are scarce *and* CPS runs high would
produce a pooled trend out of regional composition alone. Within a single region, every
donor contributes exactly one unit and that confound is gone — so the per-region test is
the one to trust, and the pooled one is shown only to see how far apart they land.

Regions share donors, so their *p*-values are not independent and must not be combined.
What is readable across them is reproducibility: `sign_agreement` is the fraction of regions
where a direction moves the same way.

In [ ]:
per_region = cps_trend(activity, meta, cps="CPS_Local", by="Brain Region")
consensus = cps_trend_consensus(per_region)

print(f"{per_region['Brain Region'].nunique()} regions x "
      f"{per_region['direction'].nunique()} directions tested\n")
print("Most reproducible CPS_Local trends (positive median_rho = rises with progression):")
consensus.head(15).round(3)

In [ ]:
activity

In [ ]:
# Where the pooled test and the per-region consensus disagree, the pooled one is picking up
# between-region structure -- the same reason `factor_association` is not the last word.
side_by_side = pd.DataFrame({
    "pooled_rho": pooled["rho"],
    "median_region_rho": consensus["median_rho"],
    "sign_agreement": consensus["sign_agreement"],
    "n_regions_q<0.1": consensus["n_significant"],
}).dropna()
side_by_side["sign_flip"] = np.sign(side_by_side["pooled_rho"]) != np.sign(side_by_side["median_region_rho"])
print(f"{int(side_by_side['sign_flip'].sum())} of {len(side_by_side)} directions flip sign "
      "between the pooled and the per-region view")
side_by_side.sort_values("pooled_rho", key=abs, ascending=False).head(12).round(3)

In [ ]:
# Per-region rho for the directions with the most reproducible trend. Read down a column:
# a row of one colour is a direction every region agrees on; a mixed row is a regional effect.
TOP = 20
top_directions = consensus.head(TOP).index
grid = (per_region.pivot(index="direction", columns="Brain Region", values="rho")
        .loc[top_directions])
q_grid = per_region.pivot(index="direction", columns="Brain Region", values="q").loc[top_directions]
# np.where, not DataFrame.replace({True: "*", ...}): replacing on an all-boolean frame
# raises IndexError inside the block manager on pandas 3.0.5.
stars = pd.DataFrame(np.where(q_grid < 0.1, "*", ""), index=q_grid.index, columns=q_grid.columns)

# Regions differ several-fold in donor count, and q is computed within a region, so where
# a star *can* appear is partly a question of n -- label the axis with it.
units = per_region.groupby("Brain Region", observed=True)["n_units"].first()
grid.columns = [f"{region}\n(n={units[region]})" for region in grid.columns]
stars.columns = grid.columns

fig, ax = plt.subplots(figsize=(0.62 * grid.shape[1] + 3.5, 0.34 * grid.shape[0] + 2))
sns.heatmap(grid, cmap="vlag", center=0, vmin=-0.8, vmax=0.8, annot=stars, fmt="",
            linewidths=0.3, cbar_kws={"label": "Spearman $\\rho$ vs CPS_Local"}, ax=ax)
ax.set_title(f"factor direction vs CPS_Local, within region\n"
             "* marks q < 0.1 within that region", fontsize=10)
ax.set_xlabel("")
ax.set_ylabel("")
fig.tight_layout()

In [ ]:
# The leading direction, drawn. One point per donor x region, sized by nuclei behind it --
# the small points are the noisy ones the correlation should not be resting on.
LEAD = consensus.index[0]
frame = meta.assign(activity=activity[LEAD]).reset_index()

fig, ax = plt.subplots(figsize=(6, 4.2))
sns.scatterplot(frame, x="CPS_Local", y="activity", hue="Brain Region", size="n_cells",
                sizes=(8, 120), alpha=0.75, ax=ax)
sns.regplot(frame, x="CPS_Local", y="activity", scatter=False, lowess=True,
            color="0.25", line_kws={"lw": 1.5}, ax=ax)
ax.set_ylabel(f"mean {LEAD} activity")
ax.set_title(f"{LEAD}: median regional $\\rho$ = {consensus.loc[LEAD, 'median_rho']:.2f}, "
             f"same sign in {consensus.loc[LEAD, 'sign_agreement']:.0%} of regions", fontsize=10)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7, frameon=False)
fig.tight_layout()

### 2b. Pan-region: one estimate per direction, donors as units

The per-region test spends its resolution on anatomy — ten answers, none of them using
more than ~50 donors. The pan-region question is the complement: across the whole cohort,
does a direction move with pseudo-progression at all?

The naive way to ask is the pooled test, and the cell above showed why it cannot be
trusted. `pan_region_units` gets there differently: z-score each direction **within its
region**, so nothing is scored high merely because a donor was sampled in regions where it
runs hot, then average those z-scores per donor with one region one vote. That leaves 84
rows — one per donor, mutually independent — so the Spearman below needs no cluster
correction and its *n* is simply the donor count.

The score has to change with the unit: `CPS_Global` is the donor-level pseudo-progression
score, and the within-donor, between-region contrast that `CPS_Local` carries is exactly
what paid for removing the region effect.

In [ ]:
# Region-adjust, then collapse to donors. Two steps, and each one is load-bearing:
# z-score every direction *within its region*, so nothing is scored high merely because a
# donor was sampled where it runs hot; then average those z-scores per donor with one
# region one vote, since weighting by nuclei would let DFC, the deepest-sampled region,
# decide the pan-region answer on its own.
blocks = pd.Series(meta.index.get_level_values("Brain Region"), index=meta.index)
donors = pd.Series(meta.index.get_level_values("Donor ID"), index=meta.index)

by_region = activity.groupby(blocks, observed=True)
spread = by_region.transform("std")
adjusted = (activity - by_region.transform("mean")) / spread.mask(spread == 0)
pan_activity = adjusted.groupby(donors, observed=True).mean()

by_donor = meta.groupby(donors, observed=True)
pan_meta = pd.DataFrame(
    {"n_units": by_donor.size(), "n_cells": by_donor["n_cells"].sum()}
).join(by_donor[["CPS_Global", "CPS_Global_ABeta", "CPS_Global_pTau",
                 "Sex", "Age at Death"]].first())

print(f"{len(pan_activity)} donors, {pan_meta['n_units'].min()}-{pan_meta['n_units'].max()} "
      f"regions each (median {pan_meta['n_units'].median():.0f})")

# CPS_Global, not CPS_Local: the unit is now the donor, and the within-donor contrast that
# CPS_Local carries is exactly what paid for removing the region effect.
pan = cps_trend(pan_activity, pan_meta, cps="CPS_Global").set_index("direction")
print(f"{int((pan['q'] < 0.1).sum())} of {len(pan)} directions reach q < 0.1\n")
pan.head(15).round(4)

In [ ]:
# Three routes to one question, so the orderings can be checked against each other: the
# per-region consensus, this donor-level test, and the naive pooled test section 2 warned
# about. The first two are independent of each other; the third is the control.
routes = pd.DataFrame({
    "pan_region_rho": pan["rho"],
    "pan_region_q": pan["q"],
    "median_region_rho": consensus["median_rho"],
    "sign_agreement": consensus["sign_agreement"],
    "pooled_rho": pooled["rho"],
}).dropna()

for label, column in [("per-region consensus", "median_region_rho"), ("naive pooled", "pooled_rho")]:
    rho = routes[["pan_region_rho", column]].corr(method="spearman").iloc[0, 1]
    print(f"Spearman between the pan-region and {label} orderings: {rho:+.3f}")
routes.sort_values("pan_region_q").head(12).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 4.8))
ax.axhline(0, color="0.85", lw=0.8)
ax.axvline(0, color="0.85", lw=0.8)
lim = 1.08 * routes[["pan_region_rho", "median_region_rho"]].abs().to_numpy().max()
ax.plot([-lim, lim], [-lim, lim], ls="--", color="0.75", lw=0.9, zorder=0)
points = ax.scatter(routes["median_region_rho"], routes["pan_region_rho"],
                    c=routes["sign_agreement"], cmap="viridis", vmin=0.5, vmax=1.0,
                    s=36, edgecolor="white", linewidth=0.4)
for name in routes["pan_region_q"].nsmallest(4).index:
    ax.annotate(name, (routes.loc[name, "median_region_rho"], routes.loc[name, "pan_region_rho"]),
                fontsize=7, xytext=(4, 3), textcoords="offset points")
fig.colorbar(points, ax=ax, label="sign agreement across regions")
ax.set(xlim=(-lim, lim), ylim=(-lim, lim),
       xlabel=r"median $\rho$ within region (CPS_Local)",
       ylabel=r"pan-region $\rho$, donors as units (CPS_Global)")
ax.set_title("Two independent routes to the same ordering", fontsize=10)
fig.tight_layout()

Whatever else it is, a pan-region test is a **between-donor** test: one unit per donor
means any donor-level variable correlated with CPS can produce the trend. Sex is the first
to rule out — `DR 41` is this fit's sex factor (`XIST`/`UTY`; see the README's covariate
table) and it appears near the top of the ranking above.

In [ ]:
from scipy import stats

donor_covariates = pan_meta.assign(
    **{"is male": (pan_meta["Sex"].astype(str) == "Male").astype(float)}
)
for name in ("is male", "Age at Death"):
    result = stats.spearmanr(donor_covariates[name], donor_covariates["CPS_Global"],
                             nan_policy="omit")
    print(f"CPS_Global vs {name:13s}: rho={result.statistic:+.3f}, p={result.pvalue:.3f}")

# cps_trend is a signed Spearman of every direction against one unit-level numeric column;
# handing it a covariate instead of a score is the same computation, on the same 84 donors.
confounders = pd.DataFrame({
    name: cps_trend(pan_activity, donor_covariates, cps=name).set_index("direction")["rho"]
    for name in ("CPS_Global", "is male", "Age at Death")
})
print("\nA direction whose |is male| rivals its |CPS_Global| is not evidence of progression:")
confounders.reindex(pan.index).head(12).round(3)

### 2c. Every donor × region as its own observation

The third unit of analysis, and the one that uses the most rows: pool all 516 donor ×
region units and correlate directly, letting every region a donor contributed count as a
separate observation. Sections 2 and 2b both set this view aside; it belongs here in full,
because it is the version with the most power and the one most often reported, and because
its two weaknesses are separable — one of them is repairable and the other is not.

**Pseudoreplication is repairable.** 516 rows come from 84 donors, so the rows are not
independent and the textbook *p* counts 516 pieces of evidence where there are 84. A
donor-level cluster bootstrap propagates the real unit of replication into the interval:
resample donors with replacement, recompute the correlation on whatever units those donors
bring, and read the percentile interval. `ci_low`/`ci_high` and `robust` come from that;
`p_naive`/`q_naive` are kept beside them, named for what they are, so the size of the gap
is visible rather than hidden.

**The region confound is not repairable at this unit.** Pooling still mixes a
between-region contrast with a between-donor one, and no resampling scheme touches that —
it is a property of the design, not of the inference. The last cell of this section
measures how much it moves each direction.

All three `CPS_Local` scores go through it.

In [ ]:
LOCAL_SCORES = {"CPS_Local": "composite", "CPS_Local_ABeta": "ABeta",
                "CPS_Local_pTau": "pTau"}
N_BOOT = 1000

clustered = {
    score: cps_trend_clustered(activity, meta, cps=score, cluster="Donor ID",
                                 n_boot=N_BOOT, random_state=0).set_index("direction")
    for score in LOCAL_SCORES
}

unit_level = pd.concat(
    {LOCAL_SCORES[score]: table[["rho", "ci_low", "ci_high", "q_naive", "robust"]]
     for score, table in clustered.items()},
    axis=1,
)
unit_level.columns = [f"{label}_{stat}" for label, stat in unit_level.columns]

counts = pd.DataFrame({
    "naive q < 0.1": {s: int((t["q_naive"] < 0.1).sum()) for s, t in clustered.items()},
    "bootstrap CI clear of 0": {s: int(t["robust"].sum()) for s, t in clustered.items()},
    "naive-only (CI straddles 0)": {
        s: int(((t["q_naive"] < 0.1) & ~t["robust"]).sum()) for s, t in clustered.items()},
    "median |rho|": {s: round(float(t["rho"].abs().median()), 3) for s, t in clustered.items()},
}).rename(index=LOCAL_SCORES)
print(f"{len(activity)} units from {meta.index.get_level_values('Donor ID').nunique()} "
      f"donors, {activity.shape[1]} live directions, {N_BOOT} donor resamples\n")
print(counts.to_string())
print("\nStrongest composite associations, with all three scores side by side:")
order = clustered["CPS_Local"]["rho"].abs().sort_values(ascending=False).index
unit_level.reindex(order).head(12).round(3)

In [ ]:
# Dot-and-whisker: the pooled rho for each score with its donor-bootstrap interval. A
# whisker crossing the dashed zero line is a direction the donor structure does not
# support, however small its naive q.
SHOW = 18
shown = list(clustered["CPS_Local"]["rho"].abs().sort_values(ascending=False).index[:SHOW])[::-1]
palette = {"CPS_Local": "#33456b", "CPS_Local_ABeta": "#d1913c", "CPS_Local_pTau": "#9a4c68"}
offsets = {"CPS_Local": 0.24, "CPS_Local_ABeta": 0.0, "CPS_Local_pTau": -0.24}

fig, ax = plt.subplots(figsize=(7.2, 0.42 * SHOW + 1.8))
ax.axvline(0, color="0.55", lw=0.9, ls="--")
base = np.arange(len(shown))
for score, table in clustered.items():
    rows = table.reindex(shown)
    y = base + offsets[score]
    ax.hlines(y, rows["ci_low"], rows["ci_high"], color=palette[score], lw=1.4, alpha=0.85)
    ax.scatter(rows["rho"], y, s=26, color=palette[score], zorder=3,
               edgecolor="white", linewidth=0.4, label=LOCAL_SCORES[score])
ax.set(yticks=base, yticklabels=shown, ylim=(-0.7, len(shown) - 0.3),
       xlabel=r"Spearman $\rho$ over 516 donor $\times$ region units (95% donor-bootstrap CI)")
ax.set_title("Every region its own observation: factor directions vs the CPS_Local scores",
             fontsize=10)
ax.legend(fontsize=8, frameon=False, loc="lower right", title="score", title_fontsize=8)
ax.grid(axis="x", color="0.92", lw=0.7)
ax.set_axisbelow(True)
fig.tight_layout()

In [ ]:
# What pooling buys, and what it costs. `gain` is how much |rho| grows relative to the
# per-region consensus; eta^2 is how much of a direction's variance across these same
# units is explained by Brain Region alone -- computed with bincount rather than a
# per-group loop, which is what makes the same expression usable at 902 library levels.
region_codes = pd.Categorical(meta.index.get_level_values("Brain Region"))
codes, n_groups = region_codes.codes.astype(np.intp), len(region_codes.categories)
unit_activity = activity.to_numpy(dtype=float)
group_sizes = np.bincount(codes, minlength=n_groups).astype(float)
populated = group_sizes > 0
grand_mean = unit_activity.mean(axis=0)
total_ss = ((unit_activity - grand_mean) ** 2).sum(axis=0)

eta2 = np.full(unit_activity.shape[1], np.nan)
for column in range(unit_activity.shape[1]):
    if total_ss[column] == 0:
        continue
    sums = np.bincount(codes, weights=unit_activity[:, column], minlength=n_groups)
    group_means = sums[populated] / group_sizes[populated]
    eta2[column] = (
        group_sizes[populated] * (group_means - grand_mean[column]) ** 2
    ).sum() / total_ss[column]
eta2_region = pd.Series(eta2, index=activity.columns, name="eta2_region")

units_vs_rest = pd.DataFrame({
    "unit_rho": clustered["CPS_Local"]["rho"],
    "robust": clustered["CPS_Local"]["robust"],
    "median_region_rho": consensus["median_rho"],
    "pan_region_rho": pan["rho"],
    "eta2_region": eta2_region,
}).dropna()
units_vs_rest["gain"] = units_vs_rest["unit_rho"].abs() - units_vs_rest["median_region_rho"].abs()
flips = np.sign(units_vs_rest["unit_rho"]) != np.sign(units_vs_rest["median_region_rho"])

print(f"{int(flips.sum())} of {len(units_vs_rest)} directions change sign between this "
      "pooled view and the per-region consensus")
print("Spearman(gain from pooling, eta^2 vs Brain Region) = "
      f"{units_vs_rest[['gain', 'eta2_region']].corr(method='spearman').iloc[0, 1]:+.3f}")
top_pooled = units_vs_rest["unit_rho"].abs().nlargest(8).index
top_region = units_vs_rest["median_region_rho"].abs().nlargest(8).index
print(f"median eta^2 vs region -- top 8 by pooled rho: "
      f"{units_vs_rest.loc[top_pooled, 'eta2_region'].median():.3f}; "
      f"top 8 by per-region rho: {units_vs_rest.loc[top_region, 'eta2_region'].median():.3f}; "
      f"all directions: {units_vs_rest['eta2_region'].median():.3f}")

fig, ax = plt.subplots(figsize=(5.6, 4.8))
ax.axhline(0, color="0.85", lw=0.8)
ax.axvline(0, color="0.85", lw=0.8)
lim = 1.08 * units_vs_rest[["unit_rho", "median_region_rho"]].abs().to_numpy().max()
ax.plot([-lim, lim], [-lim, lim], ls="--", color="0.75", lw=0.9, zorder=0)
points = ax.scatter(units_vs_rest["median_region_rho"], units_vs_rest["unit_rho"],
                    c=units_vs_rest["eta2_region"], cmap="magma_r", vmin=0, vmax=0.8,
                    s=38, edgecolor="white", linewidth=0.4)
for name in units_vs_rest.loc[flips, "unit_rho"].abs().nlargest(5).index:
    ax.annotate(name, (units_vs_rest.loc[name, "median_region_rho"],
                       units_vs_rest.loc[name, "unit_rho"]),
                fontsize=7, xytext=(4, 3), textcoords="offset points")
fig.colorbar(points, ax=ax, label=r"$\eta^2$ of the direction vs Brain Region")
ax.set(xlim=(-lim, lim), ylim=(-lim, lim),
       xlabel=r"median $\rho$ within region (section 2)",
       ylabel=r"$\rho$ over all 516 units (this section)")
ax.set_title("Off-diagonal points are what pooling adds", fontsize=10)
fig.tight_layout()

The three units of analysis do not rank the directions the same way, and the difference is
not noise: directions change sign between the pooled and the per-region view, and the ones
pooling promotes carry more region structure than average. The pooled column is the right
one to quote for "does this direction covary with pathology anywhere in this dataset"; it
is the wrong one for "does this direction track pathology *within* a region", which is what
sections 2 and 2b answer and what the rest of this notebook builds on.

The donor bootstrap is worth reading on its own terms too. Every score has directions that
clear a naive q < 0.1 and still have an interval straddling zero — associations resting on
a few deeply-sampled donors rather than on the cohort.

### Is it a state change, or the vulnerable cells disappearing?

A direction can track CPS without any neuron changing: if `DR n+` is simply what `Sst_23`
cells look like, and `Sst_23` is depleted as pathology accumulates, then the *mean*
activity per donor × region falls with CPS purely by composition. The two are
indistinguishable at this level of aggregation.

Adding `Supertype` to the unit key separates them. Within one region *and* one supertype,
cell identity is fixed and every donor still contributes one unit — so a trend that
survives is a change in what those cells are doing, not in how many of them there are.
Strata with fewer than 15 donors are dropped.

In [ ]:
within, within_meta = pseudobulk_directions(
    embed, keys=("Donor ID", "Brain Region", "Supertype"),
    min_cells=20, carry=("CPS_Local",),
)
stratified = cps_trend(within, within_meta, cps="CPS_Local",
                         by=["Brain Region", "Supertype"], min_units=15)
within_consensus = cps_trend_consensus(stratified)

n_strata = stratified.groupby(["Brain Region", "Supertype"], observed=True).ngroups
print(f"{len(within)} donor x region x supertype units -> {n_strata} strata with >= 15 donors")

survives = pd.DataFrame({
    "region_rho": consensus["median_rho"],
    "region_agreement": consensus["sign_agreement"],
    "within_supertype_rho": within_consensus["median_rho"],
    "within_supertype_agreement": within_consensus["sign_agreement"],
    "n_strata": within_consensus["n_strata"],
}).dropna()
survives["retained"] = (survives["within_supertype_rho"] / survives["region_rho"]).round(2)
survives.sort_values("within_supertype_agreement", ascending=False).head(15).round(3)

In [ ]:
# Composition-free (y) against the composition-inclusive estimate (x). Points on the
# diagonal are state changes; points collapsing toward y = 0 were abundance shifts.
fig, ax = plt.subplots(figsize=(5, 4.6))
ax.axhline(0, color="0.8", lw=0.8)
ax.axvline(0, color="0.8", lw=0.8)
lim = 1.05 * max(survives[["region_rho", "within_supertype_rho"]].abs().to_numpy().max(), 0.05)
ax.plot([-lim, lim], [-lim, lim], color="0.75", lw=0.8, ls="--", zorder=0)
points = ax.scatter(survives["region_rho"], survives["within_supertype_rho"],
                    c=survives["within_supertype_agreement"], cmap="viridis",
                    vmin=0.5, vmax=1.0, s=34, edgecolor="white", linewidth=0.4)
for name in survives["within_supertype_agreement"].nlargest(6).index:
    ax.annotate(name, (survives.loc[name, "region_rho"], survives.loc[name, "within_supertype_rho"]),
                fontsize=7, xytext=(4, 3), textcoords="offset points")
fig.colorbar(points, ax=ax, label="sign agreement across strata")
ax.set(xlabel="median $\\rho$ within region", ylabel="median $\\rho$ within region x supertype",
       xlim=(-lim, lim), ylim=(-lim, lim))
ax.set_title("Does the CPS trend survive fixing cell identity?", fontsize=10)
fig.tight_layout()

## 3. Which pathology — ABeta or pTau?

`CPS_Local` is a composite. SEA-AD also supplies the two components behind it,
`CPS_Local_ABeta` and `CPS_Local_pTau` (and their donor-level `CPS_Global_*` counterparts),
and a direction that tracks one but not the other is a far more specific claim than one
that tracks the composite.

That claim is only worth making if the two are separable in this cohort, so measure their
correlation before reading anything into a difference between them.

In [ ]:
present = [column for column in S.CPS_COLUMNS if column in meta]
score_correlation = meta[present].corr(method="spearman")
print(f"Spearman between the pseudo-progression scores, over the {len(meta)} "
      "donor x region units:")
score_correlation.round(3)

The two components correlate at rho ≈ 0.63 at the unit level — they share about 40% of
their rank variance and are far from interchangeable, so a direction can plausibly favour
one. But they are not independent either: a direction genuinely driven by ABeta will still
show a substantial pTau correlation, and only the *gap* between the two is evidence of
specificity. Read the difference, never either column alone.

Each score is run through both tests from sections 2 and 2b — the per-region consensus on
the `CPS_Local_*` variant, and the pan-region donor-level test on the matching
`CPS_Global_*` one.

In [ ]:
PATHOLOGY = {
    "composite": ("CPS_Local", "CPS_Global"),
    "ABeta": ("CPS_Local_ABeta", "CPS_Global_ABeta"),
    "pTau": ("CPS_Local_pTau", "CPS_Global_pTau"),
}

frames = []
for label, (local_score, donor_score) in PATHOLOGY.items():
    regional = cps_trend_consensus(
        cps_trend(activity, meta, cps=local_score, by="Brain Region")
    )
    donors = cps_trend(pan_activity, pan_meta, cps=donor_score).set_index("direction")
    frames.append(pd.DataFrame({
        f"{label}_region_rho": regional["median_rho"],
        f"{label}_pan_rho": donors["rho"],
        f"{label}_pan_q": donors["q"],
    }))

pathology = pd.concat(frames, axis=1).dropna()
pathology["abeta_minus_ptau"] = pathology["ABeta_pan_rho"] - pathology["pTau_pan_rho"]
pathology["best_q"] = pathology[[f"{k}_pan_q" for k in PATHOLOGY]].min(axis=1)

print("Most component-specific directions (largest gap between the two pan-region rhos), "
      "restricted to directions either component reaches q < 0.1 on:")
specific = pathology[pathology["best_q"] < 0.1]
specific.reindex(specific["abeta_minus_ptau"].abs().sort_values(ascending=False).index)[
    ["composite_pan_rho", "ABeta_pan_rho", "pTau_pan_rho", "abeta_minus_ptau", "best_q"]
].head(12).round(3)

In [ ]:
# Off the diagonal is component-specific; on it, the direction tracks whatever the two
# scores share. The dashed line is equality, not a fit -- and because the scores themselves
# correlate at ~0.63, points sit nearer the diagonal than true specificity would put them.
fig, ax = plt.subplots(figsize=(5.4, 5.0))
lim = 1.08 * pathology[["ABeta_pan_rho", "pTau_pan_rho"]].abs().to_numpy().max()
ax.plot([-lim, lim], [-lim, lim], ls="--", color="0.75", lw=0.9, zorder=0)
ax.axhline(0, color="0.85", lw=0.8)
ax.axvline(0, color="0.85", lw=0.8)

hit = pathology["best_q"] < 0.1
ax.scatter(pathology.loc[~hit, "ABeta_pan_rho"], pathology.loc[~hit, "pTau_pan_rho"],
           s=20, color="0.78", label="q >= 0.1 on both")
ax.scatter(pathology.loc[hit, "ABeta_pan_rho"], pathology.loc[hit, "pTau_pan_rho"],
           s=42, c=pathology.loc[hit, "composite_pan_rho"], cmap="coolwarm",
           vmin=-0.5, vmax=0.5, edgecolor="white", linewidth=0.4, label="q < 0.1")
for name in pathology.loc[hit, "abeta_minus_ptau"].abs().nlargest(6).index:
    ax.annotate(name, (pathology.loc[name, "ABeta_pan_rho"], pathology.loc[name, "pTau_pan_rho"]),
                fontsize=7, xytext=(4, 3), textcoords="offset points")
ax.set(xlim=(-lim, lim), ylim=(-lim, lim),
       xlabel=r"pan-region $\rho$ vs CPS_Global_ABeta",
       ylabel=r"pan-region $\rho$ vs CPS_Global_pTau")
ax.set_title("Amyloid or tau? (colour = composite $\\rho$)", fontsize=10)
ax.legend(fontsize=7, frameon=False, loc="upper left")
fig.tight_layout()

In [ ]:
# The scatter is lopsided, so test the lopsidedness rather than eyeballing it: across
# *every* live direction, is the tau correlation larger in magnitude than the amyloid one?
from scipy import stats

leans_ptau = pathology["pTau_pan_rho"].abs() > pathology["ABeta_pan_rho"].abs()
signed_rank = stats.wilcoxon(pathology["pTau_pan_rho"].abs(),
                             pathology["ABeta_pan_rho"].abs())
print(f"|rho| larger for pTau in {int(leans_ptau.sum())} of {len(pathology)} directions "
      f"(Wilcoxon signed-rank p = {signed_rank.pvalue:.1e})")
print("median |pan-region rho|: " + ", ".join(
    f"{label} {pathology[f'{label}_pan_rho'].abs().median():.3f}" for label in PATHOLOGY))

# Two things that would produce the same pattern without any tau specificity, both
# measurable: a score with more spread across donors correlates better with anything, and
# the composite is itself nearer one component than the other.
donor_scores = pan_meta[[donor for _, donor in PATHOLOGY.values()]]
labels = {donor: label for label, (_, donor) in PATHOLOGY.items()}
print("\ndonor-score IQR:      " + ", ".join(
    f"{labels[c]} {donor_scores[c].quantile(0.75) - donor_scores[c].quantile(0.25):.3f}"
    for c in donor_scores))
print("composite vs pTau:    "
      f"{stats.spearmanr(donor_scores['CPS_Global'], donor_scores['CPS_Global_pTau']).statistic:.3f}")
print("composite vs ABeta:   "
      f"{stats.spearmanr(donor_scores['CPS_Global'], donor_scores['CPS_Global_ABeta']).statistic:.3f}")

The lean is cohort-wide, not a property of a few directions: roughly four in five live
directions correlate more strongly with tau than with amyloid, and the signed-rank test on
the paired magnitudes is not close. The honest reading is therefore about the latent space
as a whole — **these factors track tau burden more closely than amyloid burden** — rather
than about any single direction being tau-specific.

Two alternative explanations are printed above rather than waved away. `CPS_Global_pTau`
does have a slightly wider donor-to-donor spread (IQR ~0.52 against ~0.44), which lifts
every correlation with it a little; and the composite score is itself nearer tau (rho 0.94)
than amyloid (0.88), so a direction fit to track "progression" inherits some of that. Both
shrink the effect but neither accounts for a 0.07 gap in median |rho|.

The per-direction gaps in the table above are the weaker claim and should be read with the
rho ≈ 0.63 between the two components firmly in mind — no direction here is clean evidence
of amyloid-specific or tau-specific biology on its own.

## 4. The delivered scCODA results

`/data/multiregion/pertpy/CPS_Local/` holds a finished scCODA run over all 174 supertypes
of the multiregion dataset against `CPS_Local`. It was not produced here and is not refit
here. Three artifacts, which are **not** interchangeable:

| File | What it is |
|---|---|
| `pertpy_summary_CPS_Local.<date>.csv` | the delivered effect, one value per supertype × region |
| `<class group>_Supertype_results.csv` | the full sweep — every effect re-estimated against each of the 174 possible reference cell types, over 6 covariates and 11 regions |
| `objects/<class group>_Supertype_abundances.h5ad` | scCODA's input: 907 libraries × 174 supertype counts, with `CPS_Local` in `obs` |

The three are one analysis, not three: the summary turns out to be the sweep's
region-agnostic `Global` fit, thresholded and broadcast across regions, and the abundance
object is what both were fit on. The cells below check that rather than assume it.

In the summary, three values mean three different things, which a `fillna(0)` would erase:
`NaN` is *not modelled* (no nuclei of that supertype in that region — verified against the
abundance counts), `0.0` is *modelled but not credible*, and a non-zero value is a credible
shift, negative meaning depleted as the score rises.

In [ ]:
# Three values with three meanings, which a naive fillna(0) would collapse into one:
# NaN  -- the supertype has no nuclei in that region, so it was never modelled
# 0.0  -- modelled, but scCODA did not call the effect credible
# else -- a credible shift in that supertype's share, negative meaning depleted
summary_path = sorted(S.CPS_LOCAL_DIR.glob(S.SCCODA_SUMMARY_GLOB))[-1]
summary = pd.read_csv(summary_path, index_col=0)
effects = (summary[summary["Cell Type"].isin(SUPERTYPES)]
           .pivot(index="Cell Type", columns="Region", values="Local Model"))
effects.columns.name = None

print(f"{summary_path.name}: {effects.shape[0]} supertypes x {effects.shape[1]} regions -- "
      f"{int((effects.fillna(0) != 0).to_numpy().sum())} credible, "
      f"{int((effects == 0).to_numpy().sum())} not credible, "
      f"{int(effects.isna().to_numpy().sum())} not modelled")
effects.round(2)

### That table has ten columns and does not have ten measurements

The shape invites a supertype × region heatmap. Before drawing one: independent per-region
MCMC fits cannot agree to six decimal places, and across the SST supertypes this table's
effect is a single value repeated over every region the supertype appears in — MTG,
SEA-AD's separately-fit flagship region, being the one exception. Reading it as ten
regional measurements would multiply the evidence by ten.

In [ ]:
# Is the delivered effect actually region-specific, or one value repeated? Independent
# per-region MCMC fits cannot agree to six decimal places, so a supertype whose row takes
# a single value across every region it appears in is one fit broadcast, and reading it as
# ten regional measurements would multiply the evidence by ten.
rows = {}
for supertype, values in effects.iterrows():
    present = values.dropna()
    if present.empty:
        continue
    mode = present.mode()
    shared = float(mode.iloc[0]) if len(mode) else np.nan
    rows[supertype] = {
        "n_regions": len(present),
        "n_distinct": int(present.nunique()),
        "shared_effect": shared,
        "deviating_regions": ", ".join(present.index[present != shared]) or "-",
    }
specificity = pd.DataFrame(rows).T.astype(
    {"n_regions": int, "n_distinct": int, "shared_effect": float})
specificity.index.name = "Cell Type"

print(f"{int((specificity['n_distinct'] == 1).sum())} of {len(specificity)} supertypes carry "
      "one value across every region they appear in")
print("deviating regions seen:",
      sorted({r for entry in specificity["deviating_regions"] for r in entry.split(", ")
              if r != "-"}))
specificity

So the usable scCODA quantity for the SST cohort is **one CPS_Local abundance effect per
supertype** (`shared_effect`), not a regional map. That is worth stating plainly, because
the regional resolution this notebook does have comes from section 2, which is computed
from the latent space rather than read off this table.

In [ ]:
# Significance here is the posterior inclusion probability, *not* `Final Parameter != 0`:
# in the sweep a zero marks only a reference's own self-row, so counting non-zeros would
# score every effect at 1 - 1/174 regardless of the evidence. Those self-rows are dropped
# and the threshold is applied to inclusion instead, at a cut recovered from where the
# summary's own credible/not-credible calls fall.
#
# Read in chunks: the neuronal results file is ~180 MB of which one covariate is wanted,
# and materialising all 2.0M rows to keep 0.3M is pure waste.
SWEEP_COLUMNS = ["Region", "Covariate", "Cell Type", "Final Parameter",
                 "Inclusion probability", "Reference Cell Type"]

chunks = []
for path in sorted(S.CPS_LOCAL_DIR.glob(S.SCCODA_RESULTS_GLOB)):
    for chunk in pd.read_csv(path, usecols=SWEEP_COLUMNS, chunksize=500_000):
        chunk = chunk[(chunk["Covariate"] == "CPS_Local")
                      & chunk["Cell Type"].isin(SUPERTYPES)]
        if len(chunk):
            chunks.append(chunk)

swept = pd.concat(chunks, ignore_index=True)
swept = swept[swept["Cell Type"] != swept["Reference Cell Type"]]
by_cell_region = swept.groupby(["Cell Type", "Region"], observed=True)
effect, inclusion = by_cell_region["Final Parameter"], by_cell_region["Inclusion probability"]
sweep = pd.DataFrame({
    "n_references": effect.size(),
    "median_effect": effect.median(),
    # The spread over reference choices is the uncertainty this sweep exists to expose,
    # and it is not the posterior SD of any one fit.
    "effect_p05": effect.quantile(0.05),
    "effect_p95": effect.quantile(0.95),
    "median_inclusion": inclusion.median(),
    "frac_credible": inclusion.apply(
        lambda v: float((v >= S.SCCODA_INCLUSION_THRESHOLD).mean())),
})
global_fit = sweep.xs("Global", level="Region").sort_values("median_effect")

print(f"{len(sweep)} supertype x region cells from "
      f"{int(sweep['n_references'].median())} references each")
print(f"credibility cut on median inclusion probability: {S.SCCODA_INCLUSION_THRESHOLD}")
agreement = pd.DataFrame({
    "summary": specificity["shared_effect"],
    "sweep_Global": global_fit["median_effect"],
    "median_inclusion": global_fit["median_inclusion"],
    "frac_references_credible": global_fit["frac_credible"],
})
agreement["summary_credible"] = agreement["summary"] != 0
agreement["difference"] = agreement["summary"] - agreement["sweep_Global"]
agreement.sort_values("sweep_Global").round(3)

### The summary *is* the region-agnostic fit

Read the table down the `difference` column. Wherever the summary calls an effect credible
it reproduces the sweep's `Global` estimate to within 0.016, and the supertypes it writes
as `0.0` are exactly those whose median inclusion probability falls below the cut — the
boundary runs between `Sst_9` at 0.815 (zeroed) and `Sst_22` at 0.841 (kept).

Two rows depart from that. `Sst_27-SEAAD` exists only in V1C and takes V1C's own fit
(−1.132 there, against +0.005 globally). And MTG, which the table above does not show,
matches neither the sweep's MTG rows nor `Global` — those cells come from outside this
file, so an MTG value is a different study's answer rather than a regional contrast.

So there is nothing to reconcile between the two artifacts: the summary gives the
delivered, thresholded effect, and the sweep gives the evidence behind it — the inclusion
probability, and how far the estimate travels when the reference cell type changes.

In [ ]:
# Left: the delivered per-region table -- one estimate broadcast, MTG excepted, which is
# why most rows are a flat band. Right: the region-agnostic Global fit with its evidence.
# The bar is the median effect over the 173 reference choices, the whisker their 5th-95th
# percentile, the number the median posterior inclusion probability, and a hollow bar
# falls below the credibility cut.
order = global_fit.index
fig, axes = plt.subplots(1, 2, figsize=(14.5, 0.42 * len(order) + 2.8),
                         gridspec_kw={"width_ratios": [1.25, 1]})

axes[0].set_facecolor("0.88")          # NaN = supertype absent from that region
sns.heatmap(effects.reindex(order), cmap="vlag", center=0, vmin=-2, vmax=2,
            annot=True, fmt=".2f", annot_kws={"fontsize": 6.5}, linewidths=0.3,
            cbar_kws={"label": "scCODA CPS_Local effect", "shrink": 0.6}, ax=axes[0])
axes[0].set_title("Region-specific: the delivered summary\n"
                  "grey = supertype absent from that region", fontsize=10)
axes[0].set_xlabel("")
axes[0].set_ylabel("")

credible = (global_fit["median_inclusion"] >= S.SCCODA_INCLUSION_THRESHOLD).to_numpy()
y = np.arange(len(order))
axes[1].barh(y, global_fit["median_effect"], height=0.66,
             color=np.where(credible, "#3b6ea5", "white"), edgecolor="#3b6ea5")
axes[1].hlines(y, global_fit["effect_p05"], global_fit["effect_p95"], color="0.25", lw=1.2)
for position, (_, row) in enumerate(global_fit.iterrows()):
    left = row["median_effect"] < 0
    axes[1].text(row["median_effect"] + (-0.04 if left else 0.04), position,
                 f"{row['median_inclusion']:.2f}", fontsize=7, va="center",
                 ha="right" if left else "left")
axes[1].axvline(0, color="0.6", lw=0.8)
axes[1].set(yticks=y, yticklabels=order, ylim=(len(order) - 0.5, -0.5),
            xlabel="effect on the supertype's share (median over 173 references)")
axes[1].set_title(f"Region-agnostic: the Global fit\nnumbers are posterior inclusion "
                  f"probability; hollow = below {S.SCCODA_INCLUSION_THRESHOLD}", fontsize=10)
fig.tight_layout()

In [ ]:
# An independent read on the sign, straight from scCODA's own input counts: the raw
# Spearman of each supertype's share of the neuronal composition against CPS_Local.
# No compositional model, no reference cell type, no shrinkage. It cannot replace the
# model -- shares are not independent, which is the problem scCODA exists to solve --
# but a credible effect whose raw share trends the other way should be flagged, not used.
count_frames, count_metas = [], []
for path in sorted(S.SCCODA_OBJECTS_DIR.glob(S.SCCODA_ABUNDANCE_GLOB)):
    abundance = ad.read_h5ad(path)
    frame = pd.DataFrame(np.asarray(abundance.X), index=abundance.obs_names,
                         columns=abundance.var_names)
    wanted = [column for column in frame.columns if column in set(SUPERTYPES)]
    if not wanted:
        continue
    info = abundance.obs.copy()
    # Over every supertype in the file, before restriction: scCODA fit the full
    # composition of its class group, so a share taken within the SST subset is not the
    # quantity its effects describe.
    info["total"] = frame.sum(axis=1)
    count_frames.append(frame[wanted])
    count_metas.append(info)

counts, count_meta = pd.concat(count_frames), pd.concat(count_metas)
shares = counts.div(count_meta.loc[counts.index, "total"], axis=0)

rows = []
for region, index in count_meta.groupby("Brain Region", observed=True).groups.items():
    index = counts.index.intersection(index)
    scores = pd.to_numeric(count_meta.loc[index, "CPS_Local"], errors="coerce")
    usable_libraries = index[scores.notna().to_numpy()]
    if len(usable_libraries) < 10:
        continue
    for supertype in shares.columns:
        share = shares.loc[usable_libraries, supertype]
        if share.nunique() <= 1:
            continue
        result = stats.spearmanr(share.to_numpy(), scores[usable_libraries].to_numpy())
        rows.append({"Cell Type": supertype, "Brain Region": region,
                     "n_libraries": len(usable_libraries),
                     "rho": result.statistic, "p": result.pvalue})
raw = pd.DataFrame(rows)
raw["q"] = bh_fdr(raw["p"].to_numpy())

check = (raw.groupby("Cell Type")["rho"].median().rename("median_share_rho")
         .to_frame()
         .join(specificity["shared_effect"]))
check["credible"] = check["shared_effect"] != 0
check["disagrees"] = check["credible"] & (
    np.sign(check["median_share_rho"]) != np.sign(check["shared_effect"]))
print(f"{int(check['credible'].sum())} supertypes with a credible scCODA effect; "
      f"sign disagreement with the raw shares in {int(check['disagrees'].sum())} of them")
check.sort_values("shared_effect").round(3)

## 5. Do the CPS-tracking directions mark the CPS-depleted supertypes?

This is where the three strands join. Section 2 found directions that move with CPS and
showed which of them survive fixing cell identity; section 3 gives, per supertype, how much
its abundance shifts with the same score. If a direction is mostly a marker of the
supertypes scCODA calls depleted, its supertype profile will anti-correlate with
`shared_effect` — and that is a composition story regardless of how strong its CPS trend
looked. A direction whose CPS trend survives section 2's within-supertype control *and*
shows no such profile correlation is the interesting case.

One limit on the x axis: scCODA writes exactly `0.0` where it did not call an effect
credible, so `shared_effect` is censored at zero rather than measured there. Nine of the
nineteen supertypes sit at that value, which flattens the correlation toward zero — read
a strong x as evidence, a weak x as the absence of it.

In [ ]:
# Mean activity per supertype, z-scored down each column so a globally strong direction
# does not dominate the comparison with scCODA's effects.
profile = split_by_sign(embed).groupby(embed.obs["Supertype"], observed=True).mean()
profile = (profile - profile.mean()) / profile.std(ddof=0).mask(lambda s: s == 0)

vulnerability = specificity["shared_effect"].reindex(profile.index)
usable = vulnerability.notna()
print(f"{int(usable.sum())} supertypes with an scCODA effect, "
      f"{int((vulnerability[usable] != 0).sum())} of them credible")

link = pd.DataFrame({
    "profile_vs_vulnerability": profile[usable.to_numpy()].corrwith(
        vulnerability[usable], method="spearman"
    ),
    "region_rho": consensus["median_rho"],
    "within_supertype_rho": within_consensus["median_rho"],
    "within_supertype_agreement": within_consensus["sign_agreement"],
}).dropna()
link["composition_driven"] = link["profile_vs_vulnerability"].abs() > 0.5
link.sort_values("within_supertype_agreement", ascending=False).head(15).round(3)

In [ ]:
# x: does the direction mark the supertypes scCODA calls depleted?
# y: does its CPS trend survive holding supertype fixed?
# Bottom-left / bottom-right split the two explanations apart.
fig, ax = plt.subplots(figsize=(5.6, 4.6))
ax.axhline(0, color="0.8", lw=0.8)
ax.axvline(0, color="0.8", lw=0.8)
points = ax.scatter(link["profile_vs_vulnerability"], link["within_supertype_rho"],
                    c=link["within_supertype_agreement"], cmap="viridis", vmin=0.5, vmax=1.0,
                    s=36, edgecolor="white", linewidth=0.4)
for name in link["within_supertype_rho"].abs().nlargest(5).index:
    ax.annotate(name, (link.loc[name, "profile_vs_vulnerability"], link.loc[name, "within_supertype_rho"]),
                fontsize=7, xytext=(4, 3), textcoords="offset points")
fig.colorbar(points, ax=ax, label="sign agreement across strata")
ax.set(xlabel="$\\rho$(supertype profile, scCODA CPS effect)",
       ylabel="median $\\rho$ vs CPS_Local, supertype fixed")
ax.set_title("Composition marker, or state change?", fontsize=10)
fig.tight_layout()

In [ ]:
# The supertype profile of the directions with the most reproducible within-supertype trend,
# with scCODA's abundance effect on the same rows for comparison.
SHOW = 12
chosen = link.sort_values("within_supertype_agreement", ascending=False).head(SHOW).index
panel = profile[chosen]

fig, axes = plt.subplots(1, 2, figsize=(0.55 * SHOW + 6, 0.36 * len(panel) + 2.2),
                         gridspec_kw={"width_ratios": [SHOW, 1.0]}, sharey=True)
sns.heatmap(panel, cmap="rocket_r", linewidths=0.3, ax=axes[0],
            cbar_kws={"label": "mean activity (z across supertypes)"})
axes[0].set_title("Which supertypes carry each direction", fontsize=10)
axes[0].set_xlabel("")
axes[0].set_ylabel("")
sns.heatmap(vulnerability.to_frame("scCODA\nCPS effect"), cmap="vlag", center=0,
            annot=True, fmt=".2f", annot_kws={"fontsize": 8}, linewidths=0.3, ax=axes[1],
            cbar=False)
axes[1].set_title("abundance", fontsize=10)
axes[1].set_ylabel("")
fig.tight_layout()

## 6. Summary

Fill in after looking at the plots above:

- Directions whose `CPS_Local` trend reproduces across regions (section 2):
- Directions the pan-region test confirms over 84 donors, and any it contradicts:
- Whether the cohort-wide tau lean in section 3 survives your own look at the two
  alternative explanations printed there (score spread, composite leaning tau):
- Individual directions that separate ABeta from pTau, given the two scores correlate
  at rho ~ 0.63 and no single direction is clean evidence on its own:
- Of those, the ones that survive fixing supertype — candidate **state** changes:
- The ones that collapse — abundance shifts wearing a factor's clothes:
- SST supertypes scCODA calls depleted with pseudo-progression, and whether the raw shares agree:
- Whether any direction links the two: strong within-supertype trend *and* no correlation with the vulnerability profile:

Caveats that belong next to any of the above:

- The scCODA effects are one value per supertype, not per region (section 3); the regional
  resolution in section 2 comes from the latent space alone.
- `CPS_Local` is donor × region, so the honest *n* is 516 units over 84 donors — and
  regions share donors, which is why no *p*-value is combined across them.
- The pan-region test is a between-donor test, so a donor-level confounder (sex, age)
  can carry it; the confounder table in section 2 is the check, not an afterthought.
- The covariate fits are decoder-only, so `q(z|x)` never saw the nuisance covariates and a
  direction can still carry technical structure; check the candidate directions against
  section 6 of `01_qc.ipynb` before reading biology into them.

In [ ]:
from session_info2 import session_info

session_info(dependencies=True)